In [1]:
import bw2data   
import bw2io

In [2]:
bw2data.projects.set_current('ei311')
len(list(bw2data.methods))

633

## step1: extract full set of flows used in the LCIA package (based on biosphere3) and do the mapping


In [3]:
from bw2io.package import BW2Package

def extract_lcia_flows_from_package(pkg_path ):
    objs = BW2Package.load_file(pkg_path)
    flow_keys = set()
    for obj in objs:
        for (key, val) in obj["data"]:
            db, code = key
            flow_keys.add((db, code))
    return flow_keys

pkg_flows = extract_lcia_flows_from_package(
    "dpGWP100.bw2package"
)
len(pkg_flows), list(pkg_flows)[:10]

(307,
 [('biosphere3', '818cee9e-231c-4b53-8ed2-47a0001802d5'),
  ('biosphere3', '4f7ebc48-dbe2-4dd9-be92-e81f22520694'),
  ('biosphere3', '33681770-a0e1-4ce8-93c3-941fd607fa5f'),
  ('biosphere3', '9ea11aea-7cb7-4b38-b936-a42c2a4bac97'),
  ('biosphere3', '50f528ca-45e7-42d0-b399-998ea63ddabf'),
  ('biosphere3', 'b018c97d-8fea-4258-b4d3-6a65669c77f4'),
  ('biosphere3', '6c977009-5c4e-4901-a4c1-ab20389cb972'),
  ('biosphere3', '468bebea-17ee-49d3-b11d-c603d91c67c8'),
  ('biosphere3', 'e42a8077-1363-40a9-a943-cfcf78e6b52e'),
  ('biosphere3', 'c8e5538f-17a1-4f62-aa64-39a91adb5cd3')])

In [4]:
newbio = bw2data.Database("ecoinvent-3.11-biosphere")
mapping = {}
missing = []

for (db, code) in pkg_flows:
    old_f = code
    # need more quick search if pkg_flows explode, worked here for my little N = 300 LCIA  
    candidates = [x for x in newbio if str(x.as_dict()['code']) == str(old_f)]

    if len(candidates) == 1:
        mapping[(db, code)] = candidates[0]
    else:
        missing.append(((db, code), candidates))

In [5]:
len(mapping), len(missing)

(260, 47)

## step2: write BW2 method 

In [6]:
from bw2io.package import BW2Package
from bw2data import Method

pkg_path = "dpGWP100.bw2package"    
objs = BW2Package.load_file(pkg_path)

print(len(objs))

# just try with one LCIA 
obj = objs[0]

print("Original method name in package:", obj["metadata"]['abbreviation'] )
print("Number of CF rows in package:", len(obj["data"]))


18
Original method name in package: climate-change-prospective-gwp100s2030p.c734b4ca3384d8a49ed136adaede30dd
Number of CF rows in package: 307


In [7]:
obj['name']

('Climate Change prospective GWP100',
 'SSP119',
 '2030',
 'pGWP100 with fixed AGWPCO2')

In [8]:
for obj in objs: 
    print(obj["name"])
    new_data = []
    dropped = 0
    
    for key, cf in obj["data"]:
        # key is of form ('biosphere3', 'uuid')
        if key in mapping:
            new_key = mapping[key]          # ('ecoinvent-3.12-biosphere', 'new_uuid')
            new_data.append((new_key, cf))  # keep the same CF value
        else:
            dropped += 1
    
    print(f"CF rows kept (mapped): {len(new_data)}")
    print(f"CF rows dropped (unmapped): {dropped}")
    
    # use the same method name
    new_method_name = obj['name']
    m = Method(new_method_name)
    m.register()
    m.write(new_data)


('Climate Change prospective GWP100', 'SSP119', '2030', 'pGWP100 with fixed AGWPCO2')
CF rows kept (mapped): 260
CF rows dropped (unmapped): 47
('Climate Change prospective GWP100', 'SSP119', '2030', 'pGWP100 with dpAGWPCO2')
CF rows kept (mapped): 260
CF rows dropped (unmapped): 47
('Climate Change prospective GWP100', 'SSP119', '2040', 'pGWP100 with fixed AGWPCO2')
CF rows kept (mapped): 260
CF rows dropped (unmapped): 47
('Climate Change prospective GWP100', 'SSP119', '2040', 'pGWP100 with dpAGWPCO2')
CF rows kept (mapped): 260
CF rows dropped (unmapped): 47
('Climate Change prospective GWP100', 'SSP119', '2050', 'pGWP100 with fixed AGWPCO2')
CF rows kept (mapped): 260
CF rows dropped (unmapped): 47
('Climate Change prospective GWP100', 'SSP119', '2050', 'pGWP100 with dpAGWPCO2')
CF rows kept (mapped): 260
CF rows dropped (unmapped): 47
('Climate Change prospective GWP100', 'SSP245', '2030', 'pGWP100 with fixed AGWPCO2')
CF rows kept (mapped): 260
CF rows dropped (unmapped): 47
('Cl

## quick lca calc

In [9]:
print([m for m in bw2data.methods if 'pGWP100' in str(m) ] )
len([m for m in bw2data.methods if 'pGWP100' in str(m) ] )


[('Climate Change prospective GWP100', 'SSP119', '2030', 'pGWP100 with fixed AGWPCO2'), ('Climate Change prospective GWP100', 'SSP119', '2030', 'pGWP100 with dpAGWPCO2'), ('Climate Change prospective GWP100', 'SSP119', '2040', 'pGWP100 with fixed AGWPCO2'), ('Climate Change prospective GWP100', 'SSP119', '2040', 'pGWP100 with dpAGWPCO2'), ('Climate Change prospective GWP100', 'SSP119', '2050', 'pGWP100 with fixed AGWPCO2'), ('Climate Change prospective GWP100', 'SSP119', '2050', 'pGWP100 with dpAGWPCO2'), ('Climate Change prospective GWP100', 'SSP245', '2030', 'pGWP100 with fixed AGWPCO2'), ('Climate Change prospective GWP100', 'SSP245', '2030', 'pGWP100 with dpAGWPCO2'), ('Climate Change prospective GWP100', 'SSP245', '2040', 'pGWP100 with fixed AGWPCO2'), ('Climate Change prospective GWP100', 'SSP245', '2040', 'pGWP100 with dpAGWPCO2'), ('Climate Change prospective GWP100', 'SSP245', '2050', 'pGWP100 with fixed AGWPCO2'), ('Climate Change prospective GWP100', 'SSP245', '2050', 'pGWP1

18

In [10]:
act = bw2data.Database('ecoinvent-3.11-cutoff').random()
act

'clear-cutting, primary forest to arable land, pasture, man made' (kilogram, BR-PI, None)

In [11]:
import bw2calc

for mm in [ ('Climate Change prospective GWP100', 'SSP585', '2030', 'pGWP100 with dpAGWPCO2'),
            ('Climate Change prospective GWP100', 'SSP585', '2030',  'pGWP100 with fixed AGWPCO2'),
            ('Climate Change prospective GWP100', 'SSP119', '2040', 'pGWP100 with dpAGWPCO2'),
            ('Climate Change prospective GWP100',  'SSP119', '2040',  'pGWP100 with fixed AGWPCO2'),
          ]: ##('IPCC 2021 - dpCFsSSP119_MY2030 - year100', 'climate change', 'pGWP100 with dp-AGWPCO2'), 
    cc =  bw2calc.lca.LCA( demand = {act:1}, method=   mm  ) 
    cc.lci()
    cc.lcia()
    print(mm, cc.score)

('Climate Change prospective GWP100', 'SSP585', '2030', 'pGWP100 with dpAGWPCO2') 10.82563684850098
('Climate Change prospective GWP100', 'SSP585', '2030', 'pGWP100 with fixed AGWPCO2') 11.818003530324829
('Climate Change prospective GWP100', 'SSP119', '2040', 'pGWP100 with dpAGWPCO2') 10.884235929513142
('Climate Change prospective GWP100', 'SSP119', '2040', 'pGWP100 with fixed AGWPCO2') 12.201079405352996


In [13]:
cc22 =  bw2calc.lca.LCA( demand = {act:1}, method=   ('ecoinvent-3.11',
  'IPCC 2021 (incl. biogenic CO2)',
  'climate change: total (incl. biogenic CO2, incl. SLCFs)',
  'global warming potential (GWP100)')   ) 
cc22.lci()
cc22.lcia()
cc22.score

10.974750706115067